In [1]:
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
import math
import seaborn as sns
import time


from kan_convolutional.KANLinear import KANLinear
from kan_convolutional.KANConv import KAN_Convolutional_Layer
from kan_convolutional import convolution 

In [2]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Device configuration: use GPU if available, else fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Malimg(Dataset):
    def __init__(self, root_dirs, transform=None):
        self.transform = transform
        self.image_files = []
        self.labels = []
        self.class_names = []

        for root_dir in root_dirs:
            for label, subfolder in enumerate(os.listdir(root_dir)):
                subfolder_path = os.path.join(root_dir, subfolder)
                if os.path.isdir(subfolder_path):
                    if subfolder not in self.class_names:
                        self.class_names.append(subfolder)
                    label = self.class_names.index(subfolder)
                    for img_file in os.listdir(subfolder_path):
                        if img_file.endswith('.png'):
                            self.image_files.append(os.path.join(subfolder_path, img_file))
                            self.labels.append(label)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = Image.open(img_name).convert('L')
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, label

    def get_class_names(self):
        return self.class_names

# Root directory for dataset
root_dirs = [
    # "C:\\Users\\Anurag Dutta\\Desktop\\Essentials\\Research\\malware\\gan_generated",
    "C:\\Users\\Rajat S Chakraborty\\Desktop\\workingr8now\\256\\celeb\\aug"
]

# Transformations
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Initialize the dataset
dataset = Malimg(root_dirs=root_dirs, transform=transform)

# Split dataset into train and test
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Now, when you train your model, make sure to move both model and data to the device (GPU or CPU).
# Example:
# model = YourModel().to(device)

# When loading data in the loop, make sure to move the data to the device
# Example:
# for images, labels in train_loader:
#     images, labels = images.to(device), labels.to(device)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class NormalizedConvolutionalKAN(nn.Module):
    def __init__(self):
        super(NormalizedConvolutionalKAN, self).__init__()
        
        self.conv1 = nn.Conv2d(1, 8, kernel_size=2, padding=1)
        self.bn1 = nn.BatchNorm2d(8)
        
        self.conv2 = nn.Conv2d(8, 16, kernel_size=2, padding=1)
        self.bn2 = nn.BatchNorm2d(16)

        self.maxpool = nn.MaxPool2d(kernel_size=(2, 2))

        self.flatten = nn.Flatten()

        self.kan1 = KANLinear(
            in_features=16 * 16 * 16,
            out_features=2,
            grid_size=10,
            spline_order=3,
            scale_noise=0.01,
            scale_base=1,
            scale_spline=1,
            base_activation=nn.SiLU,
            grid_eps=0.02,
            grid_range=[0, 1]
        )

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.maxpool(x)
        
        x = self.flatten(x)
        x = self.kan1(x)
        
        x = F.log_softmax(x, dim=1)  

        return x

# Initialize the model
model = NormalizedConvolutionalKAN()
model = model.to(device)

# # Example input tensor (move it to the correct device)
# dummy_input = torch.randn(1, 1, 128, 128).to(device)

# # Setup for mixed precision
# scaler = GradScaler()  # Gradient scaler helps in handling the dynamic range of gradients in mixed precision

# # Forward pass using mixed precision
# model.train()  # Set the model to training mode

# # Use autocast to automatically use mixed precision
# with autocast():
#     output = model(dummy_input)

# # Print the model and output devices
# print(f"Model is on: {next(model.parameters()).device}")  # Where model is located
# print(f"Output is on: {output.device}")  # Where output tensor is located

# If you were training, here's how you would use the scaler
# optimizer = ...  # Define your optimizer
# loss = ...  # Compute loss

# optimizer.zero_grad()
# scaler.scale(loss).backward()  # Scales the loss and backward pass
# scaler.step(optimizer)  # Updates the optimizer
# scaler.update()  # Updates the scaler

In [4]:
import time
import torch
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NormalizedConvolutionalKAN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

start_time = time.time()

for epoch in range(10): 
    epoch_start_time = time.time()
    model.train()
    running_loss = 0.0

    with tqdm(train_loader, unit="batch") as tepoch:
        for images, labels in tepoch:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            tepoch.set_description(f"Epoch [{epoch+1}/10]")
            tepoch.set_postfix(loss=running_loss / len(tepoch))

    epoch_time = time.time() - epoch_start_time
    print(f'Epoch [{epoch + 1}/10], Loss: {running_loss / len(train_loader):.4f}, Time elapsed: {epoch_time:.2f} seconds')

total_time = time.time() - start_time
print(f"Training completed in: {total_time:.2f} seconds")

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())  
        all_labels.extend(labels.cpu().numpy())  

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')

print(f'Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')

torch.save(model.state_dict(), 'kan_c_bn.pth')

Epoch [1/10]: 100%|██████████| 9232/9232 [39:10<00:00,  3.93batch/s, loss=0.453]  


Epoch [1/10], Loss: 0.4530, Time elapsed: 2350.31 seconds


Epoch [2/10]: 100%|██████████| 9232/9232 [41:26<00:00,  3.71batch/s, loss=0.411] 


Epoch [2/10], Loss: 0.4114, Time elapsed: 2486.84 seconds


Epoch [3/10]: 100%|██████████| 9232/9232 [41:23<00:00,  3.72batch/s, loss=0.396] 


...

Epoch [8/10], Loss: 0.3673, Time elapsed: 2467.48 seconds


Epoch [9/10]: 100%|██████████| 9232/9232 [41:10<00:00,  3.74batch/s, loss=0.364] 


Epoch [9/10], Loss: 0.3644, Time elapsed: 2470.51 seconds


Epoch [10/10]: 100%|██████████| 9232/9232 [41:35<00:00,  3.70batch/s, loss=0.359] 


Epoch [10/10], Loss: 0.3586, Time elapsed: 2495.96 seconds
Training completed in: 24666.67 seconds
Accuracy: 0.8279, Precision: 0.8285, Recall: 0.8270, F1 Score: 0.8274


In [5]:
from sklearn.metrics import confusion_matrix

# Compute the confusion matrix
conf_matrix = confusion_matrix(all_labels, all_preds)

# Print the confusion matrix
print("Confusion Matrix:")
print(conf_matrix)

Confusion Matrix:
[[65222 11184]
 [14240 57065]]
